In [ ]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
from transformers import (
    BertTokenizer,
    BertForMaskedLM,
    BertConfig,
    LineByLineTextDataset,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments
)
import torch
from sklearn.model_selection import train_test_split
from typing import Literal
from torch import nn
from transformers import BertModel,AutoModel
from huggingface_hub import hf_hub_download
from typing import Literal
import json

In [ ]:
import torch
from transformers import pipeline

pipeline = pipeline(task="text-generation", model="openai-community/gpt2", torch_dtype=torch.float16, device=0)
pipeline("Hello, I'm a language model")

In [ ]:
nbs = pd.read_csv('/Users/ritikrmohapatra/Documents/GitHub/Multi-Model-Bias-Detection-and-Debiasing-the-News/EDA/No_Corrupted_NBS.csv')

In [ ]:
 pip install -q -U google-genai

In [ ]:
nbs.isna().sum()

In [ ]:
nbs.head()

In [ ]:
nbs.shape[0]

In [ ]:
nbs['multimodal_label_y'].value_counts()

In [ ]:
import random

index = random.randint(1, 2185)
print(index)


In [ ]:
import requests


Random Selection

In [ ]:
image_path = nbs['mac_image_path'][index]
image_caption = nbs['content'][index]
article = nbs['image_description_x'][index]
with open(image_path, 'rb') as f:
    image_bytes = f.read()

In [ ]:
image_path

In [ ]:

biased_headlin = nbs[nbs['multimodal_label_y'] == 'Likely']['headline']
Non_biased_headlin = nbs[nbs['multimodal_label_y'] == 'Unlikely']['headline']

rand_idx_1 = random.randint(0, len(biased_headlin)-1)
rand_idx_2 = random.randint(0, len(Non_biased_headlin)-1)
rand_idx_3 = random.randint(0, len(Non_biased_headlin)-1)
rand_idx_4 = random.randint(0, len(biased_headlin)-1)

In [ ]:
#https://huggingface.co/docs/transformers/main/en/generation_strategies


exam_sig = f"""
Uniased: " {Non_biased_headlin.iloc[rand_idx_2]} "
Biased: "{biased_headlin.iloc[rand_idx_1]} " """ 

exam_multi = f"""
Unbiased: "{Non_biased_headlin.iloc[rand_idx_2]}"
Biased: "{biased_headlin.iloc[rand_idx_4]}"

Unbiased: "{Non_biased_headlin.iloc[rand_idx_3]}"
Biased: "{biased_headlin.iloc[rand_idx_1]}"
"""

def build_prompt(prompt_type=str):
    if prompt_type == "zero":
        return f"""Image caption: {image_caption}
Article: {article}

Rewrite the article and caption in a neutral and unbiased tone.
"""
    elif prompt_type == "single":
        return f"""Image caption: {image_caption}
Article: {article}

Here is an example of debiasing:
{exam_sig}

Now, rewrite the article and caption above in a neutral and unbiased tone.
"""
    elif prompt_type == "multi":
        return f"""Image caption: {image_caption}
Article: {article}

Here are examples of rewriting biased text into neutral form:
{exam_multi}

Now rewrite the article and caption above in the same neutral style.
"""
    elif prompt_type == "role":
        return f"""You are a professional news editor who is neutral in every scenario.Your task is to remove emotionally charged or biased language and rewrite the following article and caption in a strictly neutral,unbiased and factual tone.

Image caption: {image_caption}
Article: {article}
"""
    else:
        raise ValueError("Invalid prompt_type")


In [ ]:
!pip install google.generativeai

Gemini

In [ ]:
#https://ai.google.dev/gemini-api/docs/multimodal
from google.generativeai import GenerativeModel, types
import requests

def gemini_lm(prompt_type):
    prompt = build_prompt(prompt_type)
    contents = [
        types.Part.from_bytes(data=image_bytes, mime_type="image/jpeg"),
        prompt
    ]
    model = GenerativeModel("gemini-1.5-pro")
    response = model.generate_content(contents)
    return response.text



Llama 2

In [ ]:
#https://huggingface.co/meta-llama
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "meta-llama/Llama-2-13b-chat-hf"
hf_token = ""  

tokenizer = AutoTokenizer.from_pretrained(model_name, use_auth_token=hf_token)
model = AutoModelForCausalLM.from_pretrained(model_name, use_auth_token=hf_token, device_map="auto", torch_dtype=torch.float16)
def llama_lm(prompt_type):
    prompt = build_prompt(prompt_type)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output = model.generate(**inputs, max_length=512)
    return tokenizer.decode(output[0], skip_special_tokens=True)



DeepSeek API


In [ ]:
#https://huggingface.co/deepseek-ai
model_name = "deepseek-ai/deepseek-llm-7b-chat"
tokenizer = AutoTokenizer.from_pretrained(model_name)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = AutoModelForCausalLM.from_pretrained(model_name)
model.to(device)
def run_deepseek(prompt_type):
    prompt = build_prompt(prompt_type)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output = model.generate(**inputs, max_length=512)
    return tokenizer.decode(output[0], skip_special_tokens=True)



gpt 2

In [ ]:
#https://huggingface.co/gpt2
from transformers import GPT2Tokenizer, GPT2LMHeadModel

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")

def run_gpt2(prompt_type):
    prompt = build_prompt(prompt_type)
    inputs = tokenizer(prompt, return_tensors="pt")
    output = model.generate(**inputs, max_new_tokens=200)
    return tokenizer.decode(output[0], skip_special_tokens=True)



In [ ]:
run_gpt2('zero')

In [ ]:
df_cosine = pd.DataFrame({
    'model': ['Llama', 'GPT2', 'Gemini', 'Deepseek']
})
df_cosine['zero']=0
df_cosine['single']=0
df_cosine['multi']=0
df_cosine['role']=0
df_cosine

In [ ]:
df_model_test = pd.DataFrame({
    'model': ['Llama', 'GPT2', 'Gemini', 'Deepseek']
})
df_model_test['zero']=0
df_model_test['single']=0
df_model_test['multi']=0
df_model_test['role']=0
df_model_test

In [ ]:
df_grammar = pd.DataFrame({
    'model': ['Llama', 'GPT2', 'Gemini', 'Deepseek']
})
df_grammar['zero']=0
df_grammar['single']=0
df_grammar['multi']=0
df_grammar['role']=0
df_grammar

In [ ]:
text_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
image_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
MAX_LEN = 512


In [ ]:
import classes_for_multimodal_bias_classification.py

In [ ]:
Ensem_Mod=EnsembleModel(model_text_only=model_BABE
                        ,model_txt_and_img=model_NBS
                        ,valid_for_text=valid_BABE
                        ,valid_for_imgandtxt=valid_NBS
                        ,valid_loader_txt= valid_ldr_for_babe
                        ,valid_loader_textAndImage= valid_dataloader_nbs)checkpoint = torch.load("ensemble_model.pth")
Ensem_Mod.load_state_dict(checkpoint["model_state_dict"])
Ensem_Mod.eval()
